# Computing a Stokes problem

On the unit square $\Omega = [0,1]^2\subset\mathbb{R}^2$, we want to solve the following problem: $$\begin{align} 
\text{Find } (u,p) \in [H^1_{D}(\Omega)]^d \times H^1(\Omega) & \text{ so that } \\
-\Delta u + \nabla p &= f &&\text{ in }\Omega \\ 
\text{div}\,u &= 0 && \text{ in }\Omega \\
u &= \left( \begin{array}{cc} 4 y (1-y), & 0 \end{array} \right)^T && \text{ on } \Gamma_{\text{in}} = \{0\} \times [0,1]
\end{align}$$

where the space $[H^1_D(\Omega)]^d$ already incorporates the Dirichlet boundary conditions on $\Gamma_{\text{in}}$.

### Import libraries

In [ ]:
# ngsolve is the underlying finite element library
from ngsolve import *
# our module
from ngsxditto import *

### Construct the geometry

In [ ]:
maxh = 0.1
mesh = Mesh(unit_square.GenerateMesh(maxh=maxh))

from ngsolve.webgui import Draw
Draw(mesh)

### Construct the fluids discretization

In `ngsxditto` you have the possibility to construct different types of discretizations for your fluid.
We will use an $H^1$-conforming discretization which uses the Taylor-Hood element.
This results in the velocity space $V_h = [\mathbb{P}^k(\mathcal{T}_h)]^d$ and the according pressure space $Q_h = \mathbb{P}^{k-1}(\mathcal{T}_h)$.
Note that for polynomial orders $k<4$ you do not necessarily get a stable discretization on general meshes.
Therefore, we will use as a guiding example just $k=4$.

In [ ]:
order = 4

Now we will implement our discretized fluid.
First, we will create an object encapsulating the fluid's parameters, like viscosity, density, surface tension coefficient, ...

In [ ]:
fluid_params = FluidParameters(viscosity=1)

Now we want to actually set up our fluid and its boundary condition.
To this end, we will first define a `NGSolve`-`CoefficientFunction` describing the boundary conditions.

In [ ]:
uin = CF((4*y*(1-y),0)) # parabolic inflow

fluid = TaylorHood(mesh, order=order, fluid_params=fluid_params, f=CF((8, 0)))
fluid.SetBoundaryConditions(dirichlet={"left|bottom|top": uin})
fluid.InitializeSpaces()

Next up, we want to initialize and assemble the variational formulation.
To this end, we need to also pass the right hand side.
In our case, we pass $ f = (8,0)^T$ which corresponds to a flow from the left inflow to the right side.

In [ ]:
fluid.UpdateActiveDofs()
fluid.InitializeForms()

Alternatively we can combine above steps by writing: \
`fluid.Initialize(dirichlet=dirichlet, neumann={}, rhs=rhs)`

Finally, we can look solve the Stokes problem and take a look at the solution.

In [ ]:
gfup = fluid.SolveStokes()
gfu, gfp, _ = gfup.components
Draw(gfu, mesh)

You can also prescribe Neumann boundary conditions, describing a stress onto the fluid.
We will now consider the same domain $\Omega$, but with different boundary conditions: $$\begin{align}
    u_{D} &= 0 &&\text{ on } \{ 0,1 \} \times [0,1] \\
    \nabla_{\boldsymbol{n}} u &= -4x(1-x) &&\text{ on } [0,1] \times \{0\} \\
    \nabla_{\boldsymbol{n}} u &= 4x(1-x) &&\text{ on } [0,1] \times \{1\} 
\end{align}$$

In [ ]:
uD = CF((0,0))
stress = IfPos(y-0.5, CF((4*y*(1-y),0)), CF((-4*y*(1-y),0)))

This corresponds to a zero velocity on the top and bottom wall, a suction on the left side of the square, an ejection on the right hand side.
We can now update our boundary conditions to realize the explained behaviour and assemble the linear system.

In [ ]:
fluid.SetBoundaryConditions(dirichlet={"left|right": uD}, neumann={"top|bottom": stress})
fluid.InitializeSpaces()
fluid.f = CF((0,8))
fluid.InitializeForms()

Finally, we can solve the system.

In [ ]:
gfup = fluid.SolveStokes()
gfu, gfp, _ = gfup.components
Draw(gfu)